# 编排：从 issue 到合并 PR

本笔记本带着智能体走完一个贴近现实的端到端循环：读懂一份含糊的缺陷报告、定位缺陷、修复它、开一个 PR、通过 CI、处理评审意见，最后合并。真实的维护者工作流从来不是线性的，这正是本次练习的用意：智能体必须在多种不同的工具类型之间携带状态（读取 JSON、grep 代码、编辑文件、运行模拟 CLI、解析 CI 输出），同时还要从链条中途的两次意外中恢复：一次 CI 失败，以及一个要求补写 docstring 的评审机器人。

状态沿链条流动：issue 正文 → 文件路径 → 修复 diff → PR 编号 → CI 输出 → 评审评论 → 最终合并。fixture 中的 `gh-mock` CLI 把全部内容持久化在 `.gh-state/` 里，因此每一步都能看到前面步骤做了什么，在完全没有网络访问的情况下模拟出真实的 GitHub 工作流。

相比 iterate 笔记本，这里还教会你：

- **在长链条上进行多轮引导。** 会话文件系统和对话历史跨轮次持久保存，因此每条用户消息都接着上一条继续。我们利用这一点在最后核验最终状态。
- **链条中途的恢复。** 智能体必须读懂 CI 失败信息或评审评论并做出调整，而不是盲目重试。

fixture 位于 `example_data/orchestrate/`，包含一个模拟的 `gh` CLI、一个 issue JSON 文件，以及一个带有植入缺陷的 `src/` + `tests/` 布局。

In [ ]:
import io
import os
import zipfile
from pathlib import Path

from anthropic import Anthropic
from utilities import stream_until_end_turn, wait_for_idle_status

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
GH_TOKEN = os.environ.get("GITHUB_TOKEN")  # only needed for the github_repository sidebar

client = Anthropic()
FIXTURE = Path("example_data") / "orchestrate"

## 1. 打包 fixture

模拟仓库打包了若干文件：`src/url_utils.py` 里藏着真正的缺陷，`src/blog.py` 是一个调用方，让缺陷更容易被观察到，`tests/test_urls.py` 在缺陷修复前会失败，而 `gh-mock` CLI 加上 `issue_42.json` 提供了智能体将要驱动的那套类 GitHub 工作流。我们在内存中把该目录压缩为 zip，并作为单个文件资源上传；至于如何改为挂载真实的 GitHub 仓库，见文末的侧栏。

In [ ]:
buf = io.BytesIO()
with zipfile.ZipFile(buf, "w") as zf:
    for f in FIXTURE.rglob("*"):
        if f.is_file() and f.name != "README.md":
            zf.write(f, f.relative_to(FIXTURE))
buf.seek(0)
fixture_zip = client.beta.files.upload(file=("repo.zip", buf, "application/zip"))
print(f"fixture: {fixture_zip.id}")

## 2. 智能体 + 环境 + 会话

环境把 `pytest` 声明为 pip 依赖，这样智能体就能在它的 CI 循环中真正运行测试套件。这是 cookbook 中第一个需要网络访问来安装包的笔记本，因此在其余部分为 `limited` 的网络配置之外，还设置了 `allow_package_managers: True`。

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-orchestrate",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-orchestrate-issue-to-pr"},
    model=MODEL,
    system=(
        "You are a maintainer bot. You read issues via `./gh-mock`, explore "
        "the codebase, write fixes, and shepherd PRs through CI and review. "
        "When CI fails or a reviewer requests changes, read what they said "
        "and address it, don't just retry blindly.\n\n"
        "Work in /mnt/user. All gh-mock commands run from there."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

env = client.beta.environments.create(
    name="cookbook-orchestrate-env",
    config={
        "type": "cloud",
        "networking": {"type": "limited", "allow_package_managers": True},
        "packages": {"pip": ["pytest"]},
    },
)

session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[{"type": "file", "file_id": fixture_zip.id, "mount_path": "repo.zip"}],
    title="Issue #42 → PR",
)
print(f"session: {session.id}")

## 3. 运行完整链条

一条指令就启动整个循环。需要留意的两个恢复点是 CI 失败和评审评论。如果智能体的第一次修复不完整，例如只处理了 `é` 而漏掉 `ü`，`gh-mock pr checks` 会以非零退出码返回 pytest 输出，而智能体需要读取它并继续迭代。随后，一旦 CI 变绿，如果 `slugify()` 缺少 docstring，评审机器人会阻止合并，从而在最终合并步骤之前再给智能体一次调整的机会。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Unpack /mnt/session/uploads/repo.zip into /mnt/user "
                        "and ship a fix for issue #42 end-to-end. Read the "
                        "./gh-mock script first to see what subcommands it "
                        "supports; use those to view the issue, open a PR, "
                        "run CI, handle review feedback, and merge. Show me "
                        "the final PR state when you're done."
                    ),
                }
            ],
        }
    ],
)

print("=== full orchestrate chain ===")
stream_until_end_turn(client, session.id)

## 4. 多轮核验

会话是有状态的：容器文件系统和对话历史跨轮次持久保存，因此后续只需再发送一条 `user.message`。我们在这里用它独立核验最终状态：模拟 CLI 把 PR 状态持久化在 `.gh-state/pr_101.json`，所以打印该文件就是确认 PR 最终已合并、CI 为绿、且合并前至少有一次评审通过的最简单方式。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Print the contents of /mnt/user/.gh-state/pr_101.json so "
                        "I can see the final state, CI status, and reviews."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 侧栏：挂载真实的 GitHub 仓库

上面的 fixture 是模拟的，因此笔记本可以离线运行，你无需任何 GitHub 凭据就能试用。若要对真实仓库做真实工作，把上面的 `{"type": "file", ...}` 挂载换成 `{"type": "github_repository", ...}` 资源，API 会在会话开始时把仓库克隆进容器。

这与 iterate 笔记本中文件挂载所用的 `resources=` 字段相同，该列表接受混合类型，所以你也可以在同一次调用里既克隆仓库又挂载一个单独的配置文件。智能体的 bash/read/grep 工具把工作树看作普通目录；唯一的区别是克隆由 API 完成，而不是由你完成。

```python
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {
            "type": "github_repository",
            "url": "https://github.com/anthropics/claude-cookbooks",
            "mount_path": "/workspace/cookbook",
            "authorization_token": GH_TOKEN,
            "checkout": {"type": "branch", "name": "main"},
        }
    ],
    title="Repo explorer",
)
```

无论是私有仓库还是为克隆进行身份验证，都需要 `GITHUB_TOKEN`。克隆只在会话创建时发生一次；后续轮次在同一工作树上工作，不会重新克隆。

挂载仓库还多做了一件事。如果仓库根目录有 `.claude/skills/` 目录，其中的每个技能都会在会话开始时被发现并注入智能体，无需 `skills` 字段。在一个自带构建、测试和评审技能的仓库里工作的代码智能体，会以与人类贡献者相同的方式获得它们。[`CMA_use_skills_from_a_repo.ipynb`](CMA_use_skills_from_a_repo.ipynb) 讲解了这套布局以及“先读取再遵循”的协议。